In [1]:
# =========================================================
# 1. Environment Setup
# =========================================================

!nvidia-smi

!pip install -q -U \
    transformers \
    datasets \
    accelerate \
    peft \
    bitsandbytes \
    evaluate \
    rouge_score \
    sentencepiece \
    wandb

Sun Oct  4 18:57:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# =========================================================
# 2. Reproducibility and Configuration
# =========================================================

import os
import random
import numpy as np
import torch

from dataclasses import dataclass


# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.environ["PYTHONHASHSEED"] = str(SEED)


# ---------------------------------------------------------
# Project Configuration
# ---------------------------------------------------------

@dataclass
class Config:

    # Project
    PROJECT_NAME: str = "Efficient-LLM-Fine-Tuning-QLoRA"
    SEED: int = SEED

    # Dataset
    DATASET_NAME: str = "Samsung/samsum"

    # Model
    MODEL_NAME: str = "google/flan-t5-base"

    # Evaluation
    EVAL_SAMPLES: int = 300

    # Tokenization
    MAX_INPUT_LENGTH: int = 512
    MAX_TARGET_LENGTH: int = 128

    # Training
    NUM_EPOCHS: int = 2
    LEARNING_RATE: float = 2e-4
    WEIGHT_DECAY: float = 0.01
    WARMUP_RATIO: float = 0.05

    # LoRA
    LORA_R: int = 16
    LORA_ALPHA: int = 32
    LORA_DROPOUT: float = 0.05

    # Batch / Memory
    PER_DEVICE_TRAIN_BATCH_SIZE: int = 4
    PER_DEVICE_EVAL_BATCH_SIZE: int = 4
    GRADIENT_ACCUMULATION_STEPS: int = 4

    # Generation
    NUM_BEAMS: int = 4
    MAX_NEW_TOKENS: int = 128

    # Paths
    OUTPUT_DIR: str = "./outputs"
    ADAPTER_DIR: str = "./outputs/qlora_adapter"
    RESULTS_DIR: str = "./results"


CFG = Config()


# ---------------------------------------------------------
# Device Information
# ---------------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("PROJECT CONFIGURATION")
print("=" * 70)

print(f"Project       : {CFG.PROJECT_NAME}")
print(f"Dataset       : {CFG.DATASET_NAME}")
print(f"Base Model    : {CFG.MODEL_NAME}")
print(f"Seed          : {CFG.SEED}")
print(f"Device        : {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(
        f"VRAM          : "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

print("=" * 70)

PROJECT CONFIGURATION
Project       : Efficient-LLM-Fine-Tuning-QLoRA
Dataset       : Samsung/samsum
Base Model    : google/flan-t5-base
Seed          : 42
Device        : cuda
GPU           : Tesla T4
VRAM          : 14.56 GB


In [5]:
# =========================================================
# 3. Load and Inspect SAMSum Dataset
# =========================================================

from datasets import load_dataset
import numpy as np


# ---------------------------------------------------------
# Dataset Configuration
# ---------------------------------------------------------

DATASET_NAME = "knkarthick/samsum"


# ---------------------------------------------------------
# Load Dataset
# ---------------------------------------------------------

print("=" * 70)
print("LOADING SAMSum DATASET")
print("=" * 70)

dataset = load_dataset(DATASET_NAME)

print("\nDataset loaded successfully.")


# ---------------------------------------------------------
# Dataset Overview
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("SAMSum DATASET OVERVIEW")
print("=" * 70)

print(dataset)

print("\nDataset Splits:")

for split in dataset.keys():
    print(f"  {split:>12}: {len(dataset[split]):,} examples")


# ---------------------------------------------------------
# Dataset Columns
# ---------------------------------------------------------

print("\nColumns:")
print(dataset["train"].column_names)


# ---------------------------------------------------------
# Inspect Example
# ---------------------------------------------------------

example = dataset["train"][0]

print("\n" + "=" * 70)
print("SAMPLE TRAINING EXAMPLE")
print("=" * 70)

print("\nDialogue:")
print(example["dialogue"])

print("\nReference Summary:")
print(example["summary"])


# ---------------------------------------------------------
# Basic Length Statistics
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("BASIC DATASET STATISTICS")
print("=" * 70)

dialogue_lengths = [
    len(text.split())
    for text in dataset["train"]["dialogue"]
]

summary_lengths = [
    len(text.split())
    for text in dataset["train"]["summary"]
]

print(
    f"Average dialogue length : "
    f"{np.mean(dialogue_lengths):.2f} words"
)

print(
    f"Average summary length  : "
    f"{np.mean(summary_lengths):.2f} words"
)

print(
    f"Maximum dialogue length : "
    f"{np.max(dialogue_lengths):,} words"
)

print(
    f"Maximum summary length  : "
    f"{np.max(summary_lengths):,} words"
)

print("=" * 70)

LOADING SAMSum DATASET


README.md:   0%|          | 0.00/4.36k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/504k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/522k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]


Dataset loaded successfully.

SAMSum DATASET OVERVIEW
DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

Dataset Splits:
         train: 14,731 examples
    validation: 818 examples
          test: 819 examples

Columns:
['id', 'dialogue', 'summary']

SAMPLE TRAINING EXAMPLE

Dialogue:
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

Reference Summary:
Amanda baked cookies and will bring Jerry some tomorrow.

BASIC DATASET STATISTICS
Average dialogue length : 93.79 words
Average summary length  : 20.32 words
Maximum dialogue length : 803 words
Maximum summary length  : 64 words


In [6]:
# =========================================================
# 4. Load Pretrained Summarization Model & Tokenizer
# =========================================================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# ---------------------------------------------------------
# Model Configuration
# ---------------------------------------------------------

MODEL_NAME = "facebook/bart-large-cnn"


# ---------------------------------------------------------
# Load Tokenizer
# ---------------------------------------------------------

print("=" * 70)
print("LOADING TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("\nTokenizer loaded successfully.")


# ---------------------------------------------------------
# Load Pretrained Model
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("LOADING PRETRAINED MODEL")
print("=" * 70)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print("\nModel loaded successfully.")


# ---------------------------------------------------------
# Model Information
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("MODEL INFORMATION")
print("=" * 70)

print(f"Model name       : {MODEL_NAME}")
print(f"Vocabulary size  : {tokenizer.vocab_size:,}")
print(f"Model parameters : {model.num_parameters():,}")

print("\nModel configuration:")
print(f"Encoder max position embeddings : {model.config.max_position_embeddings}")
print(f"Decoder start token ID          : {model.config.decoder_start_token_id}")
print(f"EOS token ID                    : {model.config.eos_token_id}")
print(f"PAD token ID                    : {model.config.pad_token_id}")

print("=" * 70)

LOADING TOKENIZER


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]


Model loaded successfully.

MODEL INFORMATION
Model name       : facebook/bart-large-cnn
Vocabulary size  : 50,265
Model parameters : 406,290,432

Model configuration:
Encoder max position embeddings : 1024
Decoder start token ID          : 2
EOS token ID                    : 2
PAD token ID                    : 1


In [7]:
# =========================================================
# 5. Tokenization Configuration & Dataset Validation
# =========================================================

import random
import numpy as np
import torch


# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ---------------------------------------------------------
# Hardware Information
# ---------------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("TRAINING ENVIRONMENT")
print("=" * 70)

print(f"Device : {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(
        f"VRAM   : "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )


# ---------------------------------------------------------
# Tokenization Configuration
# ---------------------------------------------------------

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 96

print("\n" + "=" * 70)
print("TOKENIZATION CONFIGURATION")
print("=" * 70)

print(f"Max source length : {MAX_SOURCE_LENGTH}")
print(f"Max target length : {MAX_TARGET_LENGTH}")


# ---------------------------------------------------------
# Dataset Validation
# ---------------------------------------------------------

REQUIRED_COLUMNS = {"dialogue", "summary"}

print("\n" + "=" * 70)
print("DATASET VALIDATION")
print("=" * 70)

for split in ["train", "validation", "test"]:

    columns = set(dataset[split].column_names)

    missing_columns = REQUIRED_COLUMNS - columns

    assert not missing_columns, (
        f"Missing columns in {split}: {missing_columns}"
    )

    assert len(dataset[split]) > 0, (
        f"{split} split is empty."
    )

    print(
        f"{split:>12}: "
        f"{len(dataset[split]):,} examples ✓"
    )


# ---------------------------------------------------------
# Check Missing / Empty Values
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("DATA QUALITY CHECK")
print("=" * 70)

for split in ["train", "validation", "test"]:

    dialogues = dataset[split]["dialogue"]
    summaries = dataset[split]["summary"]

    empty_dialogues = sum(
        1 for x in dialogues
        if not isinstance(x, str) or not x.strip()
    )

    empty_summaries = sum(
        1 for x in summaries
        if not isinstance(x, str) or not x.strip()
    )

    print(f"\n{split}:")
    print(f"  Empty dialogues : {empty_dialogues}")
    print(f"  Empty summaries : {empty_summaries}")


# ---------------------------------------------------------
# Token Length Inspection
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("TOKEN LENGTH INSPECTION")
print("=" * 70)

sample_size = min(1000, len(dataset["train"]))

sample_dialogues = dataset["train"]["dialogue"][:sample_size]
sample_summaries = dataset["train"]["summary"][:sample_size]

source_token_lengths = [
    len(tokenizer.encode(text, add_special_tokens=True))
    for text in sample_dialogues
]

target_token_lengths = [
    len(tokenizer.encode(text, add_special_tokens=True))
    for text in sample_summaries
]


# ---------------------------------------------------------
# Statistics
# ---------------------------------------------------------

print("\nSource token lengths:")
print(f"  Mean   : {np.mean(source_token_lengths):.2f}")
print(f"  Median : {np.median(source_token_lengths):.2f}")
print(f"  P90    : {np.percentile(source_token_lengths, 90):.2f}")
print(f"  P95    : {np.percentile(source_token_lengths, 95):.2f}")
print(f"  Max    : {np.max(source_token_lengths)}")

print("\nTarget token lengths:")
print(f"  Mean   : {np.mean(target_token_lengths):.2f}")
print(f"  Median : {np.median(target_token_lengths):.2f}")
print(f"  P90    : {np.percentile(target_token_lengths, 90):.2f}")
print(f"  P95    : {np.percentile(target_token_lengths, 95):.2f}")
print(f"  Max    : {np.max(target_token_lengths)}")


# ---------------------------------------------------------
# Truncation Analysis
# ---------------------------------------------------------

source_truncated = sum(
    length > MAX_SOURCE_LENGTH
    for length in source_token_lengths
)

target_truncated = sum(
    length > MAX_TARGET_LENGTH
    for length in target_token_lengths
)

print("\n" + "=" * 70)
print("TRUNCATION ANALYSIS")
print("=" * 70)

print(
    f"Source examples exceeding {MAX_SOURCE_LENGTH}: "
    f"{source_truncated}/{sample_size} "
    f"({source_truncated / sample_size * 100:.2f}%)"
)

print(
    f"Target examples exceeding {MAX_TARGET_LENGTH}: "
    f"{target_truncated}/{sample_size} "
    f"({target_truncated / sample_size * 100:.2f}%)"
)

print("\n" + "=" * 70)
print("STEP 5 COMPLETED")
print("=" * 70)

TRAINING ENVIRONMENT
Device : cuda
GPU    : Tesla T4
VRAM   : 14.56 GB

TOKENIZATION CONFIGURATION
Max source length : 512
Max target length : 96

DATASET VALIDATION
       train: 14,731 examples ✓
  validation: 818 examples ✓
        test: 819 examples ✓

DATA QUALITY CHECK

train:
  Empty dialogues : 0
  Empty summaries : 0

validation:
  Empty dialogues : 0
  Empty summaries : 0

test:
  Empty dialogues : 0
  Empty summaries : 0

TOKEN LENGTH INSPECTION

Source token lengths:
  Mean   : 148.88
  Median : 122.50
  P90    : 297.20
  P95    : 359.10
  Max    : 615

Target token lengths:
  Mean   : 27.44
  Median : 24.00
  P90    : 46.00
  P95    : 56.00
  Max    : 75

TRUNCATION ANALYSIS
Source examples exceeding 512: 6/1000 (0.60%)
Target examples exceeding 96: 0/1000 (0.00%)

STEP 5 COMPLETED


In [8]:
# =========================================================
# 6. Tokenize SAMSum Dataset
# =========================================================

print("=" * 70)
print("TOKENIZING SAMSum DATASET")
print("=" * 70)


# ---------------------------------------------------------
# Preprocessing Function
# ---------------------------------------------------------

def tokenize_function(batch):

    # Tokenize source dialogues
    model_inputs = tokenizer(
        batch["dialogue"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )

    # Tokenize target summaries
    labels = tokenizer(
        text_target=batch["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


# ---------------------------------------------------------
# Apply Tokenization
# ---------------------------------------------------------

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing dataset",
)


# ---------------------------------------------------------
# Dataset Overview
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("TOKENIZED DATASET")
print("=" * 70)

print(tokenized_dataset)


# ---------------------------------------------------------
# Feature Inspection
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("TOKENIZED FEATURES")
print("=" * 70)

print(tokenized_dataset["train"].column_names)


# ---------------------------------------------------------
# Inspect Tokenized Example
# ---------------------------------------------------------

example = tokenized_dataset["train"][0]

print("\n" + "=" * 70)
print("TOKENIZED SAMPLE")
print("=" * 70)

print(f"\nInput IDs length : {len(example['input_ids'])}")
print(f"Label IDs length : {len(example['labels'])}")

print("\nInput IDs:")
print(example["input_ids"][:30])

print("\nLabel IDs:")
print(example["labels"][:30])


# ---------------------------------------------------------
# Verify Tokenization
# ---------------------------------------------------------

decoded_input = tokenizer.decode(
    example["input_ids"],
    skip_special_tokens=True
)

decoded_label = tokenizer.decode(
    example["labels"],
    skip_special_tokens=True
)

print("\n" + "=" * 70)
print("DECODED TOKENIZED SAMPLE")
print("=" * 70)

print("\nDecoded Dialogue:")
print(decoded_input)

print("\nDecoded Summary:")
print(decoded_label)


# ---------------------------------------------------------
# Final Verification
# ---------------------------------------------------------

assert "input_ids" in tokenized_dataset["train"].column_names
assert "attention_mask" in tokenized_dataset["train"].column_names
assert "labels" in tokenized_dataset["train"].column_names

print("\n" + "=" * 70)
print("STEP 6 COMPLETED SUCCESSFULLY")
print("=" * 70)

TOKENIZING SAMSum DATASET


Tokenizing dataset:   0%|          | 0/14731 [00:00<?, ? examples/s]

Tokenizing dataset:   0%|          | 0/818 [00:00<?, ? examples/s]

Tokenizing dataset:   0%|          | 0/819 [00:00<?, ? examples/s]


TOKENIZED DATASET
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})

TOKENIZED FEATURES
['input_ids', 'attention_mask', 'labels']

TOKENIZED SAMPLE

Input IDs length : 30
Label IDs length : 13

Input IDs:
[0, 10127, 5219, 35, 38, 17241, 1437, 15269, 4, 1832, 47, 236, 103, 116, 50118, 39237, 35, 9136, 328, 50118, 10127, 5219, 35, 38, 581, 836, 47, 3859, 48433, 2]

Label IDs:
[0, 10127, 5219, 17241, 15269, 8, 40, 836, 6509, 103, 3859, 4, 2]

DECODED TOKENIZED SAMPLE

Decoded Dialogue:
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

Decoded Summary:
Amanda baked cookies and will bring Jerry some tomorrow.

STEP 6 COMPLETED SUCCESSFULLY


In [12]:
# =========================================================
# 7A. Inspect Installed Transformers API
# =========================================================

import transformers
import inspect

print("=" * 70)
print("TRANSFORMERS ENVIRONMENT")
print("=" * 70)

print(f"Transformers version: {transformers.__version__}")

print("\nSeq2SeqTrainingArguments supported parameters:")

signature = inspect.signature(Seq2SeqTrainingArguments)

for name in signature.parameters:
    print(f"  ✓ {name}")

print("=" * 70)

TRANSFORMERS ENVIRONMENT
Transformers version: 5.18.0

Seq2SeqTrainingArguments supported parameters:
  ✓ output_dir
  ✓ per_device_train_batch_size
  ✓ num_train_epochs
  ✓ max_steps
  ✓ learning_rate
  ✓ lr_scheduler_type
  ✓ lr_scheduler_kwargs
  ✓ warmup_steps
  ✓ optim
  ✓ optim_args
  ✓ weight_decay
  ✓ adam_beta1
  ✓ adam_beta2
  ✓ adam_epsilon
  ✓ optim_target_modules
  ✓ gradient_accumulation_steps
  ✓ average_tokens_across_devices
  ✓ max_grad_norm
  ✓ label_smoothing_factor
  ✓ bf16
  ✓ fp16
  ✓ bf16_full_eval
  ✓ fp16_full_eval
  ✓ tf32
  ✓ gradient_checkpointing
  ✓ gradient_checkpointing_kwargs
  ✓ torch_compile
  ✓ torch_compile_backend
  ✓ torch_compile_mode
  ✓ use_liger_kernel
  ✓ liger_kernel_config
  ✓ use_cache
  ✓ neftune_noise_alpha
  ✓ torch_empty_cache_steps
  ✓ auto_find_batch_size
  ✓ logging_strategy
  ✓ logging_steps
  ✓ logging_first_step
  ✓ log_on_each_node
  ✓ logging_nan_inf_filter
  ✓ include_num_input_tokens_seen
  ✓ log_level
  ✓ log_level_replica
 

In [13]:
# =========================================================
# 7. Final Training Configuration
# Compatible with Transformers 5.18.0
# =========================================================

from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
)


# ---------------------------------------------------------
# Data Collator
# ---------------------------------------------------------

print("=" * 70)
print("CREATING DATA COLLATOR")
print("=" * 70)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

print("\nData collator created successfully.")


# ---------------------------------------------------------
# Memory Optimization
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("MEMORY OPTIMIZATION")
print("=" * 70)

model.gradient_checkpointing_enable()

# Required when using gradient checkpointing with BART
model.config.use_cache = False

print("Gradient checkpointing : ENABLED")
print("Model cache            : DISABLED")


# ---------------------------------------------------------
# Training Configuration
# ---------------------------------------------------------

OUTPUT_DIR = "./bart_samsum_finetuned"

training_args = Seq2SeqTrainingArguments(

    # -----------------------------------------------------
    # Output
    # -----------------------------------------------------
    output_dir=OUTPUT_DIR,

    # -----------------------------------------------------
    # Batch Configuration
    # -----------------------------------------------------
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=4,

    # -----------------------------------------------------
    # Optimization
    # -----------------------------------------------------
    learning_rate=5e-5,
    weight_decay=0.01,

    num_train_epochs=3,

    warmup_steps=500,

    optim="adamw_torch",

    # -----------------------------------------------------
    # Mixed Precision
    # -----------------------------------------------------
    fp16=True,

    # -----------------------------------------------------
    # Evaluation
    # -----------------------------------------------------
    eval_strategy="epoch",

    predict_with_generate=True,

    # -----------------------------------------------------
    # Generation
    # -----------------------------------------------------
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=4,

    # -----------------------------------------------------
    # Checkpoints
    # -----------------------------------------------------
    save_strategy="epoch",
    save_total_limit=2,

    # -----------------------------------------------------
    # Best Model
    # -----------------------------------------------------
    load_best_model_at_end=True,

    metric_for_best_model="rougeL",

    greater_is_better=True,

    # -----------------------------------------------------
    # Logging
    # -----------------------------------------------------
    logging_strategy="steps",
    logging_steps=100,

    # -----------------------------------------------------
    # Reproducibility
    # -----------------------------------------------------
    seed=SEED,
    data_seed=SEED,

    # -----------------------------------------------------
    # Performance
    # -----------------------------------------------------
    dataloader_num_workers=2,

    dataloader_pin_memory=True,

    # -----------------------------------------------------
    # Logging Backend
    # -----------------------------------------------------
    report_to="none",
)


# ---------------------------------------------------------
# Configuration Summary
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL TRAINING CONFIGURATION")
print("=" * 70)

print(f"Transformers version    : 5.18.0")

print(f"\nOutput directory        : {OUTPUT_DIR}")

print("\nBatch configuration:")
print(f"  Train batch / GPU    : {training_args.per_device_train_batch_size}")
print(f"  Eval batch / GPU     : {training_args.per_device_eval_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")

effective_batch_size = (
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
)

print(f"  Effective batch size  : {effective_batch_size}")

print("\nOptimization:")
print(f"  Learning rate         : {training_args.learning_rate}")
print(f"  Weight decay          : {training_args.weight_decay}")
print(f"  Epochs                : {training_args.num_train_epochs}")
print(f"  Warmup steps          : {training_args.warmup_steps}")
print(f"  Optimizer             : {training_args.optim}")

print("\nMemory optimization:")
print(f"  FP16                  : {training_args.fp16}")
print(f"  Gradient checkpointing: ENABLED")
print(f"  Dynamic padding       : ENABLED")

print("\nEvaluation:")
print(f"  Strategy              : {training_args.eval_strategy}")
print(f"  Generate predictions  : {training_args.predict_with_generate}")
print(f"  Generation beams      : {training_args.generation_num_beams}")
print(f"  Generation max length : {training_args.generation_max_length}")

print("\nCheckpointing:")
print(f"  Save strategy         : {training_args.save_strategy}")
print(f"  Best model metric     : {training_args.metric_for_best_model}")
print(f"  Load best model       : {training_args.load_best_model_at_end}")

print("\n" + "=" * 70)
print("STEP 7 COMPLETED SUCCESSFULLY")
print("=" * 70)

CREATING DATA COLLATOR

Data collator created successfully.

MEMORY OPTIMIZATION
Gradient checkpointing : ENABLED
Model cache            : DISABLED

FINAL TRAINING CONFIGURATION
Transformers version    : 5.18.0

Output directory        : ./bart_samsum_finetuned

Batch configuration:
  Train batch / GPU    : 4
  Eval batch / GPU     : 4
  Gradient accumulation: 4
  Effective batch size  : 16

Optimization:
  Learning rate         : 5e-05
  Weight decay          : 0.01
  Epochs                : 3
  Warmup steps          : 500
  Optimizer             : OptimizerNames.ADAMW_TORCH

Memory optimization:
  FP16                  : True
  Gradient checkpointing: ENABLED
  Dynamic padding       : ENABLED

Evaluation:
  Strategy              : IntervalStrategy.EPOCH
  Generate predictions  : True
  Generation beams      : 4
  Generation max length : 96

Checkpointing:
  Save strategy         : SaveStrategy.EPOCH
  Best model metric     : rougeL
  Load best model       : True

STEP 7 COMPLETED SUC

In [14]:
!pip install -q evaluate rouge_score

In [15]:
# =========================================================
# 8. ROUGE Evaluation
# =========================================================

import numpy as np
import evaluate


# ---------------------------------------------------------
# Load ROUGE
# ---------------------------------------------------------

print("=" * 70)
print("LOADING ROUGE METRICS")
print("=" * 70)

rouge = evaluate.load("rouge")

print("\nROUGE loaded successfully.")


# ---------------------------------------------------------
# Metric Function
# ---------------------------------------------------------

def compute_metrics(eval_preds):

    predictions, labels = eval_preds

    # -----------------------------------------------------
    # Handle predictions
    # -----------------------------------------------------

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # -----------------------------------------------------
    # Decode predictions
    # -----------------------------------------------------

    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    # -----------------------------------------------------
    # Replace -100 in labels
    # -----------------------------------------------------

    labels = np.where(
        labels != -100,
        labels,
        tokenizer.pad_token_id
    )

    # -----------------------------------------------------
    # Decode references
    # -----------------------------------------------------

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    # -----------------------------------------------------
    # Clean text
    # -----------------------------------------------------

    decoded_preds = [
        pred.strip()
        for pred in decoded_preds
    ]

    decoded_labels = [
        label.strip()
        for label in decoded_labels
    ]

    # -----------------------------------------------------
    # Compute ROUGE
    # -----------------------------------------------------

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    # -----------------------------------------------------
    # Convert to percentage
    # -----------------------------------------------------

    result = {
        key: round(value * 100, 4)
        for key, value in result.items()
    }

    # -----------------------------------------------------
    # Average generated length
    # -----------------------------------------------------

    prediction_lens = [
        np.count_nonzero(
            pred != tokenizer.pad_token_id
        )
        for pred in predictions
    ]

    result["gen_len"] = round(
        np.mean(prediction_lens),
        4
    )

    return result


# ---------------------------------------------------------
# Test Metric Function
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("ROUGE CONFIGURATION")
print("=" * 70)

print("ROUGE-1 : Enabled")
print("ROUGE-2 : Enabled")
print("ROUGE-L : Enabled")
print("Stemmer : Enabled")

print("\n" + "=" * 70)
print("STEP 8 COMPLETED SUCCESSFULLY")
print("=" * 70)

LOADING ROUGE METRICS



ROUGE loaded successfully.

ROUGE CONFIGURATION
ROUGE-1 : Enabled
ROUGE-2 : Enabled
ROUGE-L : Enabled
Stemmer : Enabled

STEP 8 COMPLETED SUCCESSFULLY


In [16]:
# =========================================================
# STEP 9 — CREATE SEQ2SEQ TRAINER
# =========================================================

from transformers import Seq2SeqTrainer, EarlyStoppingCallback

print("=" * 70)
print("CREATING SEQ2SEQ TRAINER")
print("=" * 70)


# ---------------------------------------------------------
# Create Trainer
# ---------------------------------------------------------

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)


# ---------------------------------------------------------
# Trainer Summary
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("TRAINER CONFIGURATION")
print("=" * 70)

print(f"Training examples    : {len(tokenized_dataset['train']):,}")
print(f"Validation examples  : {len(tokenized_dataset['validation']):,}")
print(f"Effective batch size : 16")
print(f"Epochs               : {training_args.num_train_epochs}")
print(f"Evaluation           : {training_args.eval_strategy}")
print(f"Early stopping       : Enabled")
print(f"Metric               : ROUGE-L")
print(f"Best model loading   : Enabled")

print("\n" + "=" * 70)
print("STEP 9 COMPLETED SUCCESSFULLY")
print("=" * 70)

CREATING SEQ2SEQ TRAINER

TRAINER CONFIGURATION
Training examples    : 14,731
Validation examples  : 818
Effective batch size : 16
Epochs               : 3
Evaluation           : IntervalStrategy.EPOCH
Early stopping       : Enabled
Metric               : ROUGE-L
Best model loading   : Enabled

STEP 9 COMPLETED SUCCESSFULLY


In [17]:
# =========================================================
# STEP 10 — FINE-TUNING BART-LARGE-CNN
# =========================================================

import time
import torch

print("=" * 70)
print("STARTING FINE-TUNING")
print("=" * 70)

print("\nModel          : facebook/bart-large-cnn")
print("Dataset        : SAMSum")
print("Train samples  : {:,}".format(len(tokenized_dataset["train"])))
print("Validation     : {:,}".format(len(tokenized_dataset["validation"])))
print("Epochs         : 3")
print("Effective BS   : 16")
print("Learning Rate  : 5e-5")
print("Precision      : FP16")
print("GPU            : {}".format(torch.cuda.get_device_name(0)))

print("\n" + "=" * 70)
print("TRAINING STARTED...")
print("=" * 70)

start_time = time.time()

train_result = trainer.train()

training_time = time.time() - start_time

print("\n" + "=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)

print(
    "\nTraining time: {:.2f} minutes".format(
        training_time / 60
    )
)

print("\nTraining metrics:")

for key, value in train_result.metrics.items():
    if isinstance(value, (int, float)):
        if "loss" in key:
            print(f"{key:<25}: {value:.4f}")
        elif "runtime" in key:
            print(f"{key:<25}: {value:.2f} sec")
        else:
            print(f"{key:<25}: {value}")
    else:
        print(f"{key:<25}: {value}")

print("\n" + "=" * 70)
print("STEP 10 COMPLETED SUCCESSFULLY")
print("=" * 70)

STARTING FINE-TUNING

Model          : facebook/bart-large-cnn
Dataset        : SAMSum
Train samples  : 14,731
Validation     : 818
Epochs         : 3
Effective BS   : 16
Learning Rate  : 5e-5
Precision      : FP16
GPU            : Tesla T4

TRAINING STARTED...


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,5.663306,1.448031,40.658700,20.756700,31.165900,31.139300,60.265300
2,4.328276,1.410182,41.113800,21.057800,31.548800,31.564900,59.871600
3,2.952912,1.482595,41.158900,20.908200,31.682500,31.664700,60.331300


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].



TRAINING COMPLETED

Training time: 75.04 minutes

Training metrics:
train_runtime            : 4495.07 sec
train_samples_per_second : 9.831
train_steps_per_second   : 0.615
total_flos               : 2.508068036345856e+16
train_loss               : 4.4232
epoch                    : 3.0

STEP 10 COMPLETED SUCCESSFULLY


In [19]:
# ============================================================
# STEP 11 — FINAL TEST EVALUATION
# ============================================================

print("=" * 70)
print("STEP 11 — FINAL TEST EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# Use the already-tokenized TEST dataset from the Trainer
# No re-tokenization / no retraining
# ------------------------------------------------------------
test_dataset = trainer.eval_dataset

# IMPORTANT:
# trainer.eval_dataset currently points to the validation set,
# so we need to locate the original test split already loaded
# in the notebook environment.

print("Trainer evaluation dataset:")
print(test_dataset)

STEP 11 — FINAL TEST EVALUATION
Trainer evaluation dataset:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 818
})


In [20]:
# ============================================================
# LOCATE EXISTING TEST DATASET
# ============================================================

print("=" * 70)
print("SEARCHING FOR EXISTING TEST DATASET")
print("=" * 70)

import gc

found = []

for name, obj in globals().copy().items():
    try:
        # Hugging Face DatasetDict
        if hasattr(obj, "keys") and hasattr(obj, "__getitem__"):
            keys = list(obj.keys())

            if "test" in keys:
                test_obj = obj["test"]

                print(f"\nFOUND DATASET VARIABLE: {name}")
                print(f"Splits: {keys}")
                print(f"Test samples: {len(test_obj)}")
                print(f"Features: {test_obj.column_names}")

                found.append((name, test_obj))

        # Direct Hugging Face Dataset
        elif hasattr(obj, "column_names") and len(obj) == 819:
            print(f"\nFOUND POSSIBLE TEST DATASET: {name}")
            print(f"Samples: {len(obj)}")
            print(f"Features: {obj.column_names}")

            found.append((name, obj))

    except Exception:
        pass

print("\n" + "=" * 70)
print(f"FOUND {len(found)} POSSIBLE TEST DATASET(S)")
print("=" * 70)

SEARCHING FOR EXISTING TEST DATASET

FOUND DATASET VARIABLE: dataset
Splits: ['train', 'validation', 'test']
Test samples: 819
Features: ['id', 'dialogue', 'summary']

FOUND DATASET VARIABLE: tokenized_dataset
Splits: ['train', 'validation', 'test']
Test samples: 819
Features: ['input_ids', 'attention_mask', 'labels']

FOUND 2 POSSIBLE TEST DATASET(S)


In [21]:
# ============================================================
# STEP 11 — FINAL TEST EVALUATION
# ============================================================

print("=" * 70)
print("STEP 11 — FINAL TEST EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Select the existing tokenized TEST split
# ------------------------------------------------------------
test_dataset = tokenized_dataset["test"]

print(f"\nTest samples: {len(test_dataset)}")
print(f"Features    : {test_dataset.column_names}")

# ------------------------------------------------------------
# 2. Evaluate BEST CHECKPOINT on TEST SET
# ------------------------------------------------------------
test_results = trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test"
)

# ------------------------------------------------------------
# 3. Display complete results
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("TEST RESULTS")
print("=" * 70)

for key, value in test_results.items():
    if isinstance(value, float):
        print(f"{key:30s}: {value:.4f}")
    else:
        print(f"{key:30s}: {value}")

# ------------------------------------------------------------
# 4. Display final summarization metrics
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("FINAL TEST SUMMARIZATION METRICS")
print("=" * 70)

metrics_to_show = [
    "test_rouge1",
    "test_rouge2",
    "test_rougeL",
    "test_rougeLsum",
    "test_gen_len",
]

for metric in metrics_to_show:
    if metric in test_results:
        print(f"{metric:20s}: {test_results[metric]:.4f}")

print("\n" + "=" * 70)
print("STEP 11 COMPLETED")
print("=" * 70)

STEP 11 — FINAL TEST EVALUATION

Test samples: 819
Features    : ['input_ids', 'attention_mask', 'labels']


[transformers] early stopping required metric_for_best_model, but did not find eval_rougeL so early stopping is disabled


Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
2.952912,1.524393,3,40.583800,20.054700,31.029800,31.073200,60.251500



TEST RESULTS
test_loss                     : 1.5244
test_rouge1                   : 40.5838
test_rouge2                   : 20.0547
test_rougeL                   : 31.0298
test_rougeLsum                : 31.0732
test_gen_len                  : 60.2515

FINAL TEST SUMMARIZATION METRICS
test_rouge1         : 40.5838
test_rouge2         : 20.0547
test_rougeL         : 31.0298
test_rougeLsum      : 31.0732
test_gen_len        : 60.2515

STEP 11 COMPLETED


In [22]:
# ============================================================
# STEP 12 — BASELINE vs FINE-TUNED
# ============================================================

import torch
import numpy as np
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("=" * 70)
print("STEP 12 — BASELINE vs FINE-TUNED")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load ORIGINAL BASELINE BART
# ------------------------------------------------------------
BASELINE_MODEL_NAME = "facebook/bart-large-cnn"

print("\nLoading original BART-large-CNN baseline...")

baseline_tokenizer = AutoTokenizer.from_pretrained(
    BASELINE_MODEL_NAME
)

baseline_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASELINE_MODEL_NAME
)

baseline_model.to("cuda")
baseline_model.eval()

print("Baseline model loaded successfully.")

# ------------------------------------------------------------
# 2. Test data
# ------------------------------------------------------------
raw_test = dataset["test"]

print(f"\nTest samples: {len(raw_test)}")

# ------------------------------------------------------------
# 3. Generation settings
# ------------------------------------------------------------
MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 96
NUM_BEAMS = 4

# ------------------------------------------------------------
# 4. Generate baseline summaries
# ------------------------------------------------------------
baseline_predictions = []
reference_summaries = []

print("\nGenerating baseline summaries...")
print("-" * 70)

for i in range(len(raw_test)):

    dialogue = raw_test[i]["dialogue"]
    reference = raw_test[i]["summary"]

    inputs = baseline_tokenizer(
        dialogue,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        return_tensors="pt"
    )

    inputs = {
        k: v.to("cuda")
        for k, v in inputs.items()
    }

    with torch.no_grad():
        generated_ids = baseline_model.generate(
            **inputs,
            num_beams=NUM_BEAMS,
            max_length=MAX_TARGET_LENGTH,
            early_stopping=True
        )

    prediction = baseline_tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )

    baseline_predictions.append(prediction)
    reference_summaries.append(reference)

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(raw_test)}")

print("\nBaseline generation completed.")

# ------------------------------------------------------------
# 5. Calculate baseline ROUGE
# ------------------------------------------------------------
rouge_metric = evaluate.load("rouge")

baseline_scores = rouge_metric.compute(
    predictions=baseline_predictions,
    references=reference_summaries,
    use_stemmer=True
)

# ------------------------------------------------------------
# 6. Calculate Fine-Tuned predictions
# ------------------------------------------------------------
print("\nGenerating Fine-Tuned summaries...")

fine_tuned_predictions = []

# Use the model currently loaded in trainer
fine_tuned_model = trainer.model
fine_tuned_model.eval()
fine_tuned_model.to("cuda")

# Use the original tokenizer
fine_tuned_tokenizer = tokenizer

for i in range(len(raw_test)):

    dialogue = raw_test[i]["dialogue"]

    inputs = fine_tuned_tokenizer(
        dialogue,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        return_tensors="pt"
    )

    inputs = {
        k: v.to("cuda")
        for k, v in inputs.items()
    }

    with torch.no_grad():
        generated_ids = fine_tuned_model.generate(
            **inputs,
            num_beams=NUM_BEAMS,
            max_length=MAX_TARGET_LENGTH,
            early_stopping=True
        )

    prediction = fine_tuned_tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )

    fine_tuned_predictions.append(prediction)

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(raw_test)}")

print("\nFine-Tuned generation completed.")

# ------------------------------------------------------------
# 7. Calculate Fine-Tuned ROUGE
# ------------------------------------------------------------
fine_tuned_scores = rouge_metric.compute(
    predictions=fine_tuned_predictions,
    references=reference_summaries,
    use_stemmer=True
)

# ------------------------------------------------------------
# 8. Comparison
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("BASELINE vs FINE-TUNED RESULTS")
print("=" * 70)

metrics = [
    ("ROUGE-1", "rouge1"),
    ("ROUGE-2", "rouge2"),
    ("ROUGE-L", "rougeL"),
    ("ROUGE-Lsum", "rougeLsum"),
]

print(
    f"{'Metric':<15}"
    f"{'Baseline':>15}"
    f"{'Fine-Tuned':>15}"
    f"{'Improvement':>18}"
)

print("-" * 63)

for display_name, key in metrics:

    baseline = baseline_scores[key] * 100
    fine_tuned = fine_tuned_scores[key] * 100
    improvement = fine_tuned - baseline

    print(
        f"{display_name:<15}"
        f"{baseline:>14.2f}"
        f"{fine_tuned:>15.2f}"
        f"{improvement:>+17.2f}"
    )

print("=" * 70)

# ------------------------------------------------------------
# 9. Save results in memory
# ------------------------------------------------------------
comparison_results = {
    "baseline": {
        key: value * 100
        for key, value in baseline_scores.items()
    },
    "fine_tuned": {
        key: value * 100
        for key, value in fine_tuned_scores.items()
    }
}

print("\nSTEP 12 COMPLETED")
print("=" * 70)

STEP 12 — BASELINE vs FINE-TUNED

Loading original BART-large-CNN baseline...


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Baseline model loaded successfully.

Test samples: 819

Generating baseline summaries...
----------------------------------------------------------------------
Processed 100/819
Processed 200/819
Processed 300/819
Processed 400/819
Processed 500/819
Processed 600/819
Processed 700/819
Processed 800/819

Baseline generation completed.

Generating Fine-Tuned summaries...
Processed 100/819
Processed 200/819
Processed 300/819
Processed 400/819
Processed 500/819
Processed 600/819
Processed 700/819
Processed 800/819

Fine-Tuned generation completed.

BASELINE vs FINE-TUNED RESULTS
Metric                Baseline     Fine-Tuned       Improvement
---------------------------------------------------------------
ROUGE-1                 31.03          40.58            +9.55
ROUGE-2                 10.36          20.12            +9.76
ROUGE-L                 23.34          31.02            +7.67
ROUGE-Lsum              23.34          31.05            +7.71

STEP 12 COMPLETED


In [23]:
# ============================================================
# STEP 13 — QUALITATIVE EVALUATION
# ============================================================

print("=" * 70)
print("STEP 13 — QUALITATIVE EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# Select representative test examples
# ------------------------------------------------------------

NUM_EXAMPLES = 6

# Use evenly distributed examples across the test set
indices = np.linspace(
    0,
    len(raw_test) - 1,
    NUM_EXAMPLES,
    dtype=int
)

qualitative_results = []

for idx in indices:

    dialogue = raw_test[idx]["dialogue"]
    reference = raw_test[idx]["summary"]
    baseline = baseline_predictions[idx]
    fine_tuned = fine_tuned_predictions[idx]

    qualitative_results.append({
        "index": int(idx),
        "dialogue": dialogue,
        "reference": reference,
        "baseline": baseline,
        "fine_tuned": fine_tuned
    })

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

for i, result in enumerate(qualitative_results, start=1):

    print("\n" + "=" * 70)
    print(f"EXAMPLE {i}  |  TEST INDEX: {result['index']}")
    print("=" * 70)

    print("\n[DIALOGUE]")
    print(result["dialogue"])

    print("\n[REFERENCE SUMMARY]")
    print(result["reference"])

    print("\n[BASELINE — BART-LARGE-CNN]")
    print(result["baseline"])

    print("\n[FINE-TUNED — BART-LARGE-CNN]")
    print(result["fine_tuned"])

print("\n" + "=" * 70)
print("STEP 13 COMPLETED")
print("=" * 70)

STEP 13 — QUALITATIVE EVALUATION

EXAMPLE 1  |  TEST INDEX: 0

[DIALOGUE]
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

[REFERENCE SUMMARY]
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

[BASELINE — BART-LARGE-CNN]
Hannah: Hey, do you have Betty's number? Amanda: Lemme check. Ask Larry. He called her last time we were at the park together. I don't know him well. He's very nice. I'd rather you texted him. Just text him.

[FINE-TUNED — BART-LARGE-CNN]
Hannah is looking for Betty's number but Amanda can't find it. Larry called Betty the last time they were at the park together. Hannah doesn't kn

In [24]:
# ============================================================
# STEP 14 — ERROR ANALYSIS
# ============================================================

import re
import numpy as np

print("=" * 70)
print("STEP 14 — ERROR ANALYSIS")
print("=" * 70)


# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------
def word_count(text):
    return len(text.split())


# ------------------------------------------------------------
# 1. Length statistics
# ------------------------------------------------------------
reference_lengths = np.array([
    word_count(x) for x in reference_summaries
])

prediction_lengths = np.array([
    word_count(x) for x in fine_tuned_predictions
])

length_ratio = prediction_lengths / np.maximum(reference_lengths, 1)


print("\n" + "=" * 70)
print("SUMMARY LENGTH ANALYSIS")
print("=" * 70)

print(f"Reference mean length : {reference_lengths.mean():.2f}")
print(f"Prediction mean length: {prediction_lengths.mean():.2f}")

print(f"Reference median      : {np.median(reference_lengths):.2f}")
print(f"Prediction median     : {np.median(prediction_lengths):.2f}")

print(f"Mean length ratio     : {length_ratio.mean():.2f}x")


# ------------------------------------------------------------
# 2. Detect potential over-generation
# ------------------------------------------------------------
over_generation_mask = length_ratio >= 1.8

print(
    f"\nPotential over-generation cases "
    f"(prediction >= 1.8x reference): "
    f"{over_generation_mask.sum()} / {len(raw_test)}"
)


# ------------------------------------------------------------
# 3. Detect very long predictions
# ------------------------------------------------------------
long_prediction_mask = prediction_lengths >= 80

print(
    f"Very long predictions (>= 80 words): "
    f"{long_prediction_mask.sum()} / {len(raw_test)}"
)


# ------------------------------------------------------------
# 4. Display worst length mismatches
# ------------------------------------------------------------
worst_indices = np.argsort(length_ratio)[-5:][::-1]

print("\n" + "=" * 70)
print("TOP 5 POTENTIAL OVER-GENERATION CASES")
print("=" * 70)

for rank, idx in enumerate(worst_indices, start=1):

    print("\n" + "-" * 70)
    print(f"CASE {rank} | TEST INDEX: {idx}")

    print(
        f"Reference length : {reference_lengths[idx]} words"
    )

    print(
        f"Prediction length: {prediction_lengths[idx]} words"
    )

    print(
        f"Length ratio     : {length_ratio[idx]:.2f}x"
    )

    print("\nREFERENCE:")
    print(reference_summaries[idx])

    print("\nFINE-TUNED:")
    print(fine_tuned_predictions[idx])


# ------------------------------------------------------------
# 5. Save structured error analysis
# ------------------------------------------------------------
error_analysis = {
    "reference_mean_length": float(reference_lengths.mean()),
    "prediction_mean_length": float(prediction_lengths.mean()),
    "reference_median_length": float(np.median(reference_lengths)),
    "prediction_median_length": float(np.median(prediction_lengths)),
    "mean_length_ratio": float(length_ratio.mean()),
    "over_generation_cases": int(over_generation_mask.sum()),
    "long_prediction_cases": int(long_prediction_mask.sum()),
}

print("\n" + "=" * 70)
print("ERROR ANALYSIS SUMMARY")
print("=" * 70)

for key, value in error_analysis.items():
    print(f"{key:35s}: {value}")

print("\n" + "=" * 70)
print("STEP 14 COMPLETED")
print("=" * 70)

STEP 14 — ERROR ANALYSIS

SUMMARY LENGTH ANALYSIS
Reference mean length : 20.02
Prediction mean length: 43.90
Reference median      : 18.00
Prediction median     : 44.00
Mean length ratio     : 2.89x

Potential over-generation cases (prediction >= 1.8x reference): 601 / 819
Very long predictions (>= 80 words): 0 / 819

TOP 5 POTENTIAL OVER-GENERATION CASES

----------------------------------------------------------------------
CASE 1 | TEST INDEX: 140
Reference length : 3 words
Prediction length: 40 words
Length ratio     : 13.33x

REFERENCE:
Sean overslept again.

FINE-TUNED:
Sean overslept again and Sam is not happy about it. He will try to make up for it later on. He is angry with himself for oversleeping.    Sam is also angry with him, too, because he forgot about the meeting. 

----------------------------------------------------------------------
CASE 2 | TEST INDEX: 808
Reference length : 4 words
Prediction length: 42 words
Length ratio     : 10.50x

REFERENCE:
Stan Lee is dead.

In [26]:
# ============================================================
# STEP 15 — PROFESSIONAL INFERENCE PIPELINE
# ============================================================

import re
import torch

print("=" * 70)
print("STEP 15 — PROFESSIONAL INFERENCE PIPELINE")
print("=" * 70)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

trainer.model.to(DEVICE)
trainer.model.eval()

print(f"Device: {DEVICE}")


# ------------------------------------------------------------
# Inference function
# ------------------------------------------------------------

def summarize_dialogue(
    dialogue,
    max_source_length=512,
    max_summary_length=96,
    min_summary_length=8,
    num_beams=4,
):
    """
    Generate a concise summary for a dialogue.

    Parameters
    ----------
    dialogue : str
        Input conversation.

    max_source_length : int
        Maximum number of source tokens.

    max_summary_length : int
        Maximum number of generated tokens.

    min_summary_length : int
        Minimum number of generated tokens.

    num_beams : int
        Beam search size.

    Returns
    -------
    str
        Generated summary.
    """

    # --------------------------------------------------------
    # Validate input
    # --------------------------------------------------------

    if dialogue is None:
        raise ValueError("Dialogue cannot be None.")

    if not isinstance(dialogue, str):
        dialogue = str(dialogue)

    dialogue = dialogue.strip()

    if not dialogue:
        raise ValueError("Dialogue cannot be empty.")

    # --------------------------------------------------------
    # Tokenize
    # --------------------------------------------------------

    inputs = tokenizer(
        dialogue,
        max_length=max_source_length,
        truncation=True,
        return_tensors="pt",
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    # --------------------------------------------------------
    # Generate
    # --------------------------------------------------------

    with torch.no_grad():

        generated_ids = trainer.model.generate(
            **inputs,
            max_length=max_summary_length,
            min_length=min_summary_length,
            num_beams=num_beams,
            early_stopping=True,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
        )

    # --------------------------------------------------------
    # Decode
    # --------------------------------------------------------

    summary = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    # --------------------------------------------------------
    # Basic cleanup
    # --------------------------------------------------------

    summary = re.sub(r"\s+", " ", summary).strip()

    return summary


print("\nInference function created successfully.")

print("\n" + "=" * 70)
print("STEP 15 READY")
print("=" * 70)

STEP 15 — PROFESSIONAL INFERENCE PIPELINE
Device: cuda

Inference function created successfully.

STEP 15 READY


In [27]:
# ============================================================
# STEP 15 — INFERENCE TEST
# ============================================================

test_dialogue = """
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check.
Amanda: Sorry, can't find it.
Amanda: Ask Larry.
Amanda: He called her last time we were at the park together.
Hannah: I don't know him well.
Amanda: Don't be shy, he's very nice.
Hannah: I'd rather you texted him.
Amanda: Just text him.
Hannah: Alright.
"""

summary = summarize_dialogue(test_dialogue)

print("=" * 70)
print("INPUT DIALOGUE")
print("=" * 70)
print(test_dialogue)

print("\n" + "=" * 70)
print("GENERATED SUMMARY")
print("=" * 70)
print(summary)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer RobertaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


INPUT DIALOGUE

Hannah: Hey, do you have Betty's number?
Amanda: Lemme check.
Amanda: Sorry, can't find it.
Amanda: Ask Larry.
Amanda: He called her last time we were at the park together.
Hannah: I don't know him well.
Amanda: Don't be shy, he's very nice.
Hannah: I'd rather you texted him.
Amanda: Just text him.
Hannah: Alright.


GENERATED SUMMARY
Larry called Betty the last time Amanda and Hannah were at the park together. Amanda will text him Betty's number.


In [28]:
# ============================================================
# STEP 16 — EXPORT FINAL MODEL
# ============================================================

import os
import json
import torch

print("=" * 70)
print("STEP 16 — EXPORT FINAL MODEL")
print("=" * 70)

# ------------------------------------------------------------
# Fix tokenizer warning in inference function
# ------------------------------------------------------------

def summarize_dialogue(
    dialogue,
    max_source_length=512,
    max_summary_length=96,
    min_summary_length=8,
    num_beams=4,
):
    if dialogue is None:
        raise ValueError("Dialogue cannot be None.")

    if not isinstance(dialogue, str):
        dialogue = str(dialogue)

    dialogue = dialogue.strip()

    if not dialogue:
        raise ValueError("Dialogue cannot be empty.")

    inputs = tokenizer(
        dialogue,
        max_length=max_source_length,
        truncation=True,
        return_tensors="pt",
    )

    device = next(trainer.model.parameters()).device

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        generated_ids = trainer.model.generate(
            **inputs,
            max_length=max_summary_length,
            min_length=min_summary_length,
            num_beams=num_beams,
            early_stopping=True,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
        )

    summary = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    return " ".join(summary.split())


# ------------------------------------------------------------
# Export directory
# ------------------------------------------------------------

EXPORT_DIR = "./samsum_bart_large_cnn_final"

os.makedirs(EXPORT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Save model
# ------------------------------------------------------------

trainer.model.save_pretrained(
    EXPORT_DIR,
    safe_serialization=True
)

# ------------------------------------------------------------
# Save tokenizer
# ------------------------------------------------------------

tokenizer.save_pretrained(EXPORT_DIR)

# ------------------------------------------------------------
# Save generation configuration
# ------------------------------------------------------------

generation_config = {
    "max_source_length": 512,
    "max_summary_length": 96,
    "min_summary_length": 8,
    "num_beams": 4,
    "no_repeat_ngram_size": 3,
    "length_penalty": 1.0,
    "early_stopping": True,
}

with open(
    os.path.join(EXPORT_DIR, "generation_config.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        generation_config,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# Save project metadata
# ------------------------------------------------------------

metadata = {
    "base_model": "facebook/bart-large-cnn",
    "dataset": "knkarthick/samsum",
    "task": "Dialogue Summarization",
    "max_source_length": 512,
    "max_target_length": 96,
    "test_samples": 819,
    "test_rouge1": 40.5838,
    "test_rouge2": 20.0547,
    "test_rougeL": 31.0298,
    "test_rougeLsum": 31.0732,
    "baseline_rouge1": 31.03,
    "baseline_rouge2": 10.36,
    "baseline_rougeL": 23.34,
    "fine_tuned_rouge1": 40.58,
    "fine_tuned_rouge2": 20.12,
    "fine_tuned_rougeL": 31.02,
    "training_epochs": 3,
    "learning_rate": 5e-5,
    "effective_batch_size": 16,
    "fp16": True,
    "gradient_checkpointing": True
}

with open(
    os.path.join(EXPORT_DIR, "metadata.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# Show exported files
# ------------------------------------------------------------

print("\nExported files:")

for filename in sorted(os.listdir(EXPORT_DIR)):
    path = os.path.join(EXPORT_DIR, filename)

    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 ** 2)
        print(f"  {filename:<35} {size_mb:8.2f} MB")

print("\n" + "=" * 70)
print("STEP 16 COMPLETED")
print("=" * 70)
print(f"Model exported to: {EXPORT_DIR}")
print("=" * 70)

STEP 16 — EXPORT FINAL MODEL


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Exported files:
  config.json                             0.00 MB
  generation_config.json                  0.00 MB
  metadata.json                           0.00 MB
  model.safetensors                    1550.12 MB
  tokenizer.json                          3.39 MB
  tokenizer_config.json                   0.00 MB

STEP 16 COMPLETED
Model exported to: ./samsum_bart_large_cnn_final


In [29]:
# ============================================================
# STEP 17 — PROFESSIONAL PROJECT METADATA
# ============================================================

import json
import os

print("=" * 70)
print("STEP 17 — PROFESSIONAL PROJECT METADATA")
print("=" * 70)

PROJECT_METADATA = {
    "project_name": "BART Large CNN Fine-Tuning for Dialogue Summarization",

    "task": "Abstractive Dialogue Summarization",

    "base_model": "facebook/bart-large-cnn",

    "dataset": {
        "name": "knkarthick/samsum",
        "train_samples": 14731,
        "validation_samples": 818,
        "test_samples": 819
    },

    "tokenization": {
        "max_source_length": 512,
        "max_target_length": 96,
        "dynamic_padding": True
    },

    "training": {
        "epochs": 3,
        "learning_rate": 5e-5,
        "weight_decay": 0.01,
        "per_device_train_batch_size": 4,
        "gradient_accumulation_steps": 4,
        "effective_batch_size": 16,
        "warmup_steps": 500,
        "fp16": True,
        "gradient_checkpointing": True,
        "generation_num_beams": 4
    },

    "evaluation": {
        "test_samples": 819,

        "baseline": {
            "rouge1": 31.03,
            "rouge2": 10.36,
            "rougeL": 23.34,
            "rougeLsum": 23.34
        },

        "fine_tuned": {
            "rouge1": 40.58,
            "rouge2": 20.12,
            "rougeL": 31.02,
            "rougeLsum": 31.05
        },

        "improvement": {
            "rouge1": 9.55,
            "rouge2": 9.76,
            "rougeL": 7.67,
            "rougeLsum": 7.71
        }
    },

    "error_analysis": {
        "reference_mean_length": 20.02,
        "prediction_mean_length": 43.90,
        "mean_length_ratio": 2.89,
        "potential_over_generation_cases": 601,
        "test_samples": 819
    },

    "inference": {
        "max_source_length": 512,
        "max_summary_length": 96,
        "min_summary_length": 8,
        "num_beams": 4,
        "no_repeat_ngram_size": 3,
        "length_penalty": 1.0
    },

    "hardware": {
        "training_gpu": "NVIDIA Tesla T4",
        "cuda": True
    },

    "export": {
        "format": "SafeTensors",
        "model_file": "model.safetensors",
        "approximate_model_size_gb": 1.55
    }
}

# ------------------------------------------------------------
# Save metadata
# ------------------------------------------------------------

METADATA_PATH = "./samsum_bart_large_cnn_final/project_summary.json"

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        PROJECT_METADATA,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nProject metadata saved successfully:")
print(METADATA_PATH)

print("\n" + "=" * 70)
print("STEP 17 COMPLETED")
print("=" * 70)

STEP 17 — PROFESSIONAL PROJECT METADATA

Project metadata saved successfully:
./samsum_bart_large_cnn_final/project_summary.json

STEP 17 COMPLETED


In [30]:
# ============================================================
# STEP 17B — CREATE PROFESSIONAL INFERENCE SCRIPT
# ============================================================

from pathlib import Path

EXPORT_DIR = Path("./samsum_bart_large_cnn_final")
INFERENCE_PATH = EXPORT_DIR / "inference.py"

inference_code = r'''
import re
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


MODEL_DIR = Path(__file__).resolve().parent


def load_model():
    """Load the fine-tuned BART model and tokenizer."""

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_DIR
    )

    model.to(device)
    model.eval()

    return tokenizer, model, device


def summarize_dialogue(
    dialogue,
    tokenizer,
    model,
    device,
    max_source_length=512,
    max_summary_length=96,
    min_summary_length=8,
    num_beams=4,
):
    """Generate a concise summary for a dialogue."""

    if dialogue is None:
        raise ValueError("Dialogue cannot be None.")

    if not isinstance(dialogue, str):
        dialogue = str(dialogue)

    dialogue = dialogue.strip()

    if not dialogue:
        raise ValueError("Dialogue cannot be empty.")

    inputs = tokenizer(
        dialogue,
        max_length=max_source_length,
        truncation=True,
        return_tensors="pt",
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        generated_ids = model.generate(
            **inputs,
            max_length=max_summary_length,
            min_length=min_summary_length,
            num_beams=num_beams,
            early_stopping=True,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
        )

    summary = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    return re.sub(r"\s+", " ", summary).strip()


if __name__ == "__main__":

    tokenizer, model, device = load_model()

    sample_dialogue = """
    Hannah: Do you have Betty's number?
    Amanda: I can't find it. Ask Larry.
    Hannah: I don't know Larry.
    Amanda: He's very nice.
    Hannah: Okay, I'll text him.
    """

    summary = summarize_dialogue(
        sample_dialogue,
        tokenizer,
        model,
        device,
    )

    print("\nGenerated Summary:")
    print(summary)
'''

# Fix missing Path import in generated script
inference_code = inference_code.replace(
    "import re\n",
    "import re\nfrom pathlib import Path\n"
)

with open(
    INFERENCE_PATH,
    "w",
    encoding="utf-8"
) as f:
    f.write(inference_code.strip() + "\n")

print("=" * 70)
print("STEP 17B — INFERENCE SCRIPT")
print("=" * 70)

print(f"\nCreated successfully:")
print(INFERENCE_PATH)

print("\nFile size:")
print(f"{INFERENCE_PATH.stat().st_size / 1024:.2f} KB")

print("\n" + "=" * 70)
print("STEP 17B COMPLETED")
print("=" * 70)

STEP 17B — INFERENCE SCRIPT

Created successfully:
samsum_bart_large_cnn_final/inference.py

File size:
2.17 KB

STEP 17B COMPLETED


In [31]:
# ============================================================
# STEP 17C — STANDALONE INFERENCE TEST
# ============================================================

import sys
import subprocess

print("=" * 70)
print("STEP 17C — STANDALONE INFERENCE TEST")
print("=" * 70)

result = subprocess.run(
    [
        sys.executable,
        "samsum_bart_large_cnn_final/inference.py"
    ],
    capture_output=True,
    text=True
)

print("\nSTDOUT:")
print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

print("\n" + "=" * 70)

if result.returncode == 0:
    print("STANDALONE INFERENCE TEST: PASSED")
else:
    print("STANDALONE INFERENCE TEST: FAILED")
    print(f"Return code: {result.returncode}")

print("=" * 70)

STEP 17C — STANDALONE INFERENCE TEST

STDOUT:


STDERR:
[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 4853.17it/s]
Traceback (most recent call last):
  File "/content/samsum_bart_large_cnn_final/inference.py", line 97, in <module>
    summary = summarize_dialogue(
        sample_dialogue,
    ...<2 lines>...
        device,
    )
  File "/content/samsum_bart_large_cnn_final/inference.py", line 66, in summarize_dialogue
    generated_ids = model.generate(
        **inputs,
    ...<5 lines>...
        length_penalty=1.0,
    )
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py", line 2820, in generate
    self._prepare_special_tokens(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        generation_config, kwargs_has_attention_mask, d

In [32]:
# ============================================================
# STEP 17C — FIX EXPORTED GENERATION CONFIG
# ============================================================

import json
from pathlib import Path

EXPORT_DIR = Path("./samsum_bart_large_cnn_final")

# ------------------------------------------------------------
# BART special token configuration
# ------------------------------------------------------------

generation_config = {
    "decoder_start_token_id": 2,
    "bos_token_id": 0,
    "eos_token_id": 2,
    "pad_token_id": 1,
    "forced_bos_token_id": 0,

    "max_length": 96,
    "min_length": 8,
    "num_beams": 4,
    "early_stopping": True,
    "no_repeat_ngram_size": 3,
    "length_penalty": 1.0
}

# ------------------------------------------------------------
# Save generation config
# ------------------------------------------------------------

generation_config_path = EXPORT_DIR / "generation_config.json"

with open(
    generation_config_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        generation_config,
        f,
        indent=2
    )

print("=" * 70)
print("GENERATION CONFIG FIXED")
print("=" * 70)

print("\nSaved:")
print(generation_config_path)

print("\nConfiguration:")
for key, value in generation_config.items():
    print(f"{key:30s}: {value}")

print("\n" + "=" * 70)
print("GENERATION CONFIG READY")
print("=" * 70)

GENERATION CONFIG FIXED

Saved:
samsum_bart_large_cnn_final/generation_config.json

Configuration:
decoder_start_token_id        : 2
bos_token_id                  : 0
eos_token_id                  : 2
pad_token_id                  : 1
forced_bos_token_id           : 0
max_length                    : 96
min_length                    : 8
num_beams                     : 4
early_stopping                : True
no_repeat_ngram_size          : 3
length_penalty                : 1.0

GENERATION CONFIG READY


In [33]:
# ============================================================
# STEP 17C — UPDATE INFERENCE SCRIPT
# ============================================================

from pathlib import Path

EXPORT_DIR = Path("./samsum_bart_large_cnn_final")
INFERENCE_PATH = EXPORT_DIR / "inference.py"

inference_code = r'''
import re
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


MODEL_DIR = Path(__file__).resolve().parent


def load_model():
    """Load the fine-tuned BART model and tokenizer."""

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_DIR
    )

    # Explicit BART generation configuration
    model.generation_config.decoder_start_token_id = 2
    model.generation_config.bos_token_id = 0
    model.generation_config.eos_token_id = 2
    model.generation_config.pad_token_id = 1
    model.generation_config.forced_bos_token_id = 0

    model.to(device)
    model.eval()

    return tokenizer, model, device


def summarize_dialogue(
    dialogue,
    tokenizer,
    model,
    device,
    max_source_length=512,
    max_summary_length=96,
    min_summary_length=8,
    num_beams=4,
):
    """Generate a concise summary for a dialogue."""

    if dialogue is None:
        raise ValueError("Dialogue cannot be None.")

    if not isinstance(dialogue, str):
        dialogue = str(dialogue)

    dialogue = dialogue.strip()

    if not dialogue:
        raise ValueError("Dialogue cannot be empty.")

    inputs = tokenizer(
        dialogue,
        max_length=max_source_length,
        truncation=True,
        return_tensors="pt",
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        generated_ids = model.generate(
            **inputs,
            max_length=max_summary_length,
            min_length=min_summary_length,
            num_beams=num_beams,
            early_stopping=True,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
        )

    summary = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    return re.sub(r"\s+", " ", summary).strip()


if __name__ == "__main__":

    tokenizer, model, device = load_model()

    sample_dialogue = """
    Hannah: Do you have Betty's number?
    Amanda: I can't find it. Ask Larry.
    Hannah: I don't know Larry.
    Amanda: He's very nice.
    Hannah: Okay, I'll text him.
    """

    summary = summarize_dialogue(
        sample_dialogue,
        tokenizer,
        model,
        device,
    )

    print("\nGenerated Summary:")
    print(summary)
'''

with open(
    INFERENCE_PATH,
    "w",
    encoding="utf-8"
) as f:
    f.write(inference_code.strip() + "\n")

print("=" * 70)
print("INFERENCE SCRIPT UPDATED")
print("=" * 70)

print(f"\nUpdated:")
print(INFERENCE_PATH)

print("\nGeneration configuration is now explicitly handled.")
print("=" * 70)

INFERENCE SCRIPT UPDATED

Updated:
samsum_bart_large_cnn_final/inference.py

Generation configuration is now explicitly handled.


In [34]:
# ============================================================
# STEP 17C — RETEST STANDALONE INFERENCE
# ============================================================

import sys
import subprocess

print("=" * 70)
print("STEP 17C — RETEST STANDALONE INFERENCE")
print("=" * 70)

result = subprocess.run(
    [
        sys.executable,
        "samsum_bart_large_cnn_final/inference.py"
    ],
    capture_output=True,
    text=True
)

print("\nSTDOUT:")
print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

print("\n" + "=" * 70)

if result.returncode == 0:
    print("STANDALONE INFERENCE TEST: PASSED")
else:
    print("STANDALONE INFERENCE TEST: FAILED")
    print(f"Return code: {result.returncode}")

print("=" * 70)

STEP 17C — RETEST STANDALONE INFERENCE

STDOUT:

Generated Summary:
Hannah will text Larry about Betty's number.


STDERR:
[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 4698.23it/s]


STANDALONE INFERENCE TEST: PASSED


In [35]:
# ============================================================
# STEP 18A — CREATE FINAL PORTABLE PACKAGE
# ============================================================

import os
import shutil
from pathlib import Path

print("=" * 70)
print("STEP 18A — CREATE FINAL PORTABLE PACKAGE")
print("=" * 70)

SOURCE_DIR = Path("./samsum_bart_large_cnn_final")
PACKAGE_DIR = Path("./SAMSum_BART_Summarization")

# ------------------------------------------------------------
# Create clean package structure
# ------------------------------------------------------------

if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)

PACKAGE_DIR.mkdir(parents=True)

# Model directory
model_dir = PACKAGE_DIR / "model"
model_dir.mkdir()

# Inference directory
inference_dir = PACKAGE_DIR / "inference"
inference_dir.mkdir()

# ------------------------------------------------------------
# Copy model files
# ------------------------------------------------------------

model_files = [
    "model.safetensors",
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "generation_config.json",
    "special_tokens_map.json",
    "vocab.json",
    "merges.txt",
]

for filename in model_files:

    source = SOURCE_DIR / filename

    if source.exists():
        shutil.copy2(
            source,
            model_dir / filename
        )

# ------------------------------------------------------------
# Copy inference script
# ------------------------------------------------------------

shutil.copy2(
    SOURCE_DIR / "inference.py",
    inference_dir / "inference.py"
)

# ------------------------------------------------------------
# Copy metadata
# ------------------------------------------------------------

for filename in [
    "metadata.json",
    "project_summary.json"
]:

    source = SOURCE_DIR / filename

    if source.exists():
        shutil.copy2(
            source,
            PACKAGE_DIR / filename
        )

# ------------------------------------------------------------
# Show structure
# ------------------------------------------------------------

print("\nFinal package structure:\n")

for path in sorted(PACKAGE_DIR.rglob("*")):

    if path.is_file():

        relative = path.relative_to(PACKAGE_DIR)
        size_mb = path.stat().st_size / (1024 ** 2)

        print(
            f"{str(relative):50s} "
            f"{size_mb:8.2f} MB"
        )

# ------------------------------------------------------------
# Create ZIP
# ------------------------------------------------------------

ZIP_PATH = shutil.make_archive(
    "SAMSum_BART_Summarization",
    "zip",
    root_dir=PACKAGE_DIR.parent,
    base_dir=PACKAGE_DIR.name
)

zip_size_gb = os.path.getsize(ZIP_PATH) / (1024 ** 3)

print("\n" + "=" * 70)
print("PACKAGE CREATED")
print("=" * 70)

print(f"\nZIP file:")
print(ZIP_PATH)

print(f"\nZIP size: {zip_size_gb:.2f} GB")

print("\n" + "=" * 70)
print("STEP 18A COMPLETED")
print("=" * 70)

STEP 18A — CREATE FINAL PORTABLE PACKAGE

Final package structure:

inference/inference.py                                 0.00 MB
metadata.json                                          0.00 MB
model/config.json                                      0.00 MB
model/generation_config.json                           0.00 MB
model/model.safetensors                             1550.12 MB
model/tokenizer.json                                   3.39 MB
model/tokenizer_config.json                            0.00 MB
project_summary.json                                   0.00 MB

PACKAGE CREATED

ZIP file:
/content/SAMSum_BART_Summarization.zip

ZIP size: 1.40 GB

STEP 18A COMPLETED


In [36]:
# ============================================================
# STEP 18B — DOWNLOAD FINAL PACKAGE
# ============================================================

from google.colab import files

ZIP_PATH = "/content/SAMSum_BART_Summarization.zip"

print("=" * 70)
print("STEP 18B — DOWNLOAD FINAL PACKAGE")
print("=" * 70)

print("\nStarting download...")
print("File size: ~1.40 GB")
print("\nPlease keep the browser tab open until the download finishes.")

files.download(ZIP_PATH)

STEP 18B — DOWNLOAD FINAL PACKAGE

Starting download...
File size: ~1.40 GB

Please keep the browser tab open until the download finishes.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>